# Task 1: LoRA fine-tuning of Qwen2.5-0.5B-Instruct (English reminder -> JSON)

**Goal:** turn requests like "Remind me to call mom tomorrow at 5pm" into `{"action","task","day","time"}` JSON.
**Data:** 800 synthetic training examples; 90 held-out test examples (80 standard with 10 unseen task phrases + 10 hand-written edge cases).
**Metrics:** valid JSON %, exact match % (whole JSON equal), field accuracy % (4 fields scored separately).

**Run order:** the first cell limits Kaggle to one GPU (SFTTrainer does not support DataParallel). The pip install and `torchao` uninstall cells fix library version conflicts; restart the session after them, then run from the dataset cell onward. Training uses float32 with `fp16=False, bf16=False` because the Kaggle T4/P100 does not support the bfloat16 kernel used by the newer trl version.

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate

In [ ]:
!pip uninstall -y torchao

In [3]:
import json, random
random.seed(42)

train_tasks = [
    "call mom", "buy milk", "submit the report", "pay the electricity bill",
    "book train tickets", "water the plants", "meet Rahul", "renew my passport",
    "email the professor", "go to the gym", "pick up the parcel", "recharge my phone",
    "take medicine", "finish the assignment", "call the plumber", "back up the database",
    "review the PR", "order groceries", "clean the desk", "pay credit card bill",
    "call the dentist", "send the invoice", "check flight status", "update resume",
    "join the standup", "buy coffee beans", "submit tax forms", "walk the dog"
]
# tasks never seen in training (tests generalization)
unseen_test_tasks = [
    "cancel Netflix subscription", "fertilize the balcony garden", "upload lab records",
    "book eye checkup", "return library books", "message Priya about notes",
    "calibrate the sensor", "download bank statement", "pack gym bag", "iron formal shirt"
]
days = [
    ("today","today"), ("tonight","today"), ("tomorrow","tomorrow"), ("tmrw","tomorrow"),
    ("on Monday","monday"), ("this Monday","monday"), ("on Tuesday","tuesday"),
    ("on Wednesday","wednesday"), ("on Thursday","thursday"), ("on Friday","friday"),
    ("on Saturday","saturday"), ("on Sunday","sunday"), ("next week","next week")
]
times = [
    ("5pm","17:00"), ("5:00 pm","17:00"), ("9am","09:00"), ("9:30am","09:30"),
    ("6:30pm","18:30"), ("half past 6","18:30"), ("noon","12:00"), ("12pm","12:00"),
    ("8am","08:00"), ("10:15pm","22:15"), ("7pm","19:00"), ("7 in the evening","19:00"),
    ("3:45pm","15:45"), ("14:30","14:30"), ("midnight","00:00"), ("11am","11:00")
]
templates = [
    "Remind me to {t} {d} at {tm}",
    "Set a reminder: {t}, {d} at {tm}",
    "Don't let me forget to {t} {d} at {tm}",
    "{d} at {tm} I need to {t}, remind me",
    "pls remind me {d} {tm} to {t}",
    "Can you remind me to {t} at {tm} {d}?",
    "Hey, ping me {d} around {tm} so I {t}",
    "Schedule a reminder for {tm} {d}: {t}",
    "I gotta {t} {d} at {tm} — set an alert",
    "Make sure I {t} at {tm} {d}",
    "Reminder for {d} ({tm}): {t}",
    "Alert me at {tm} {d} to {t}"
]

def make_sample(pool):
    t = random.choice(pool); d, dv = random.choice(days); tm, tv = random.choice(times)
    return {"prompt": random.choice(templates).format(t=t, d=d, tm=tm),
            "answer": json.dumps({"action":"reminder","task":t,"day":dv,"time":tv})}

train_data    = [make_sample(train_tasks) for _ in range(800)]
standard_test = [make_sample(unseen_test_tasks) for _ in range(80)]

def E(p, a, t, d, tm):
    return {"prompt": p, "answer": json.dumps({"action":a,"task":t,"day":d,"time":tm})}

edge_cases = [
    E("Actually don't remind me to call mom tomorrow at 5pm","cancel_reminder","call mom","tomorrow","17:00"),
    E("Remind me to pay rent at 9am and call Dad at 6pm tomorrow","reminder","pay rent","tomorrow","09:00"),
    E("Ping me in 2 hours to check the oven","reminder","check the oven","today","unknown"),
    E('Remind me tomorrow at 8am to read "Deep Learning" chapter 4',"reminder",'read "Deep Learning" chapter 4',"tomorrow","08:00"),
    E("Set reminder on Friday at 6:30 to buy milk","reminder","buy milk","friday","18:30"),
    E("Remind me to submit the report by EOD today","reminder","submit the report","today","17:00"),
    E("Can you remind me to call Rahul on Monday?","reminder","call Rahul","monday","unknown"),
    E("Wake me up at quarter to seven tomorrow","reminder","wake up","tomorrow","06:45"),
    E("Remind me next Tuesday at 4pm to renew license","reminder","renew license","tuesday","16:00"),
    E("Forget about reminding me to go to the gym tonight at 8pm","cancel_reminder","go to the gym","today","20:00"),
]
test_data = standard_test + edge_cases
random.shuffle(test_data)

train, test = train_data, test_data
edge_prompts = {e["prompt"] for e in edge_cases}
print(len(train), len(test)); print(train[0])

800 90
{'prompt': 'Alert me at 5pm tonight to call the dentist', 'answer': '{"action": "reminder", "task": "call the dentist", "day": "today", "time": "17:00"}'}


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32).to("cuda")

SYS = 'Convert the request to JSON with keys action, task, day, time. Output only JSON.'

def make_prompt(p):
    msgs = [{"role":"system","content":SYS},{"role":"user","content":p}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def evaluate(m):
    m.eval(); valid = exact = fok = ftotal = 0; outs = []
    for ex in test:
        inp = tok(make_prompt(ex["prompt"]), return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = m.generate(**inp, max_new_tokens=80, do_sample=False)
        text = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gold = json.loads(ex["answer"])
        try:
            pred = json.loads(text); valid += 1
            exact += (pred == gold)
            for k in gold:
                fok += (isinstance(pred, dict) and pred.get(k) == gold[k]); ftotal += 1
        except Exception:
            ftotal += len(gold)
        outs.append((ex["prompt"], text, ex["answer"]))
    n = len(test)
    print(f"valid JSON: {valid/n:.2%} | exact match: {exact/n:.2%} | field accuracy: {fok/ftotal:.2%}")
    return outs

def breakdown(outs):
    for label, keep in [("standard", lambda p: p not in edge_prompts),
                        ("edge",     lambda p: p in edge_prompts)]:
        rows = [r for r in outs if keep(r[0])]; ok = 0
        for p, t, g in rows:
            try: ok += json.loads(t) == json.loads(g)
            except Exception: pass
        print(f"  {label}: exact match {ok}/{len(rows)} = {ok/len(rows):.1%}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Baseline results (base Qwen2.5-0.5B-Instruct, before fine-tuning)
These were measured earlier with the baseline cells below, run on the same 90-example test set, greedy decoding (do_sample=False), in a session before the kernel restart needed to fix a library error. Outputs are not stored in this notebook.

| Scoring | Valid JSON | Exact match | Field accuracy |
|---|---|---|---|
| Strict (`json.loads` on raw output) | 2.22% | 0.00% (standard 0/80, edge 0/10) | 0.56% |
| Fences removed (lenient) | 100.00% | 0.00% | 18.89% |

Fine-tuned (LoRA) results are printed further down: valid JSON 100%, exact match 90.00%, field accuracy 96.39%.

In [ ]:
base_outs = evaluate(model)
breakdown(base_outs)
for p, t, g in base_outs[:3]:
    print("\nPROMPT:", p, "\nBASE OUTPUT:", t, "\nGOLD:", g)

In [ ]:
import re

def strip_fences(text):
    # remove ```json ... ``` markdown fences, keep only the JSON inside
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def lenient_score(outs):
    valid = exact = fok = ftotal = 0
    for p, t, g in outs:
        gold = json.loads(g)
        try:
            pred = json.loads(strip_fences(t)); valid += 1
            exact += (pred == gold)
            for k in gold:
                fok += (isinstance(pred, dict) and pred.get(k) == gold[k]); ftotal += 1
        except Exception:
            ftotal += len(gold)
    n = len(outs)
    print(f"[lenient] valid JSON: {valid/n:.2%} | exact match: {exact/n:.2%} | field accuracy: {fok/ftotal:.2%}")

lenient_score(base_outs)

In [5]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def format_row(ex):
    return {"messages": [
        {"role":"system","content":SYS},
        {"role":"user","content":ex["prompt"]},
        {"role":"assistant","content":ex["answer"]},
    ]}
train_ds = Dataset.from_list([format_row(x) for x in train])

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
sft_cfg = SFTConfig(
    output_dir="./qwen_reminder_lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds,
                     processing_class=tok, peft_config=lora_cfg)
trainer.model.print_trainable_parameters()
trainer.train()

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
25,0.771446
50,0.203387
75,0.169536
100,0.160462
125,0.152274
150,0.149338
175,0.149097
200,0.149353
225,0.141890
250,0.141452


TrainOutput(global_step=300, training_loss=0.20591739972432455, metrics={'train_runtime': 276.4847, 'train_samples_per_second': 8.68, 'train_steps_per_second': 1.085, 'total_flos': 440158793963520.0, 'train_loss': 0.20591739972432455, 'epoch': 3.0})

In [6]:
ft_outs = evaluate(trainer.model)
breakdown(ft_outs)

valid JSON: 100.00% | exact match: 90.00% | field accuracy: 96.39%
  standard: exact match 79/80 = 98.8%
  edge: exact match 2/10 = 20.0%


In [12]:
json.dump([list(x) for x in ft_outs], open("task1_results_tuned.json","w"), indent=2)

In [13]:
print("\n--- FINE-TUNED FAILURES ---")
for p, t, g in ft_outs:
    try: ok = json.loads(t) == json.loads(g)
    except Exception: ok = False
    if not ok:
        print(f"PROMPT: {p}\nGOLD:   {g}\nPRED:   {t}\n{'-'*40}")


--- FINE-TUNED FAILURES ---
PROMPT: Remind me tomorrow at 8am to read "Deep Learning" chapter 4
GOLD:   {"action": "reminder", "task": "read \"Deep Learning\" chapter 4", "day": "tomorrow", "time": "08:00"}
PRED:   {"action": "reminder", "task": "read 'Deep Learning' chapter 4", "day": "tomorrow", "time": "08:00"}
----------------------------------------
PROMPT: Forget about reminding me to go to the gym tonight at 8pm
GOLD:   {"action": "cancel_reminder", "task": "go to the gym", "day": "today", "time": "20:00"}
PRED:   {"action": "reminder", "task": "go to the gym", "day": "today", "time": "19:00"}
----------------------------------------
PROMPT: Remind me next Tuesday at 4pm to renew license
GOLD:   {"action": "reminder", "task": "renew license", "day": "tuesday", "time": "16:00"}
PRED:   {"action": "reminder", "task": "renew license", "day": "next tuesday", "time": "16:00"}
----------------------------------------
PROMPT: Remind me to submit the report by EOD today
GOLD:   {"actio

## LoRA settings and training result
| Setting | Value |
|---|---|
| Rank r / alpha / dropout | 16 / 32 / 0.05 |
| Target modules | q, k, v, o, gate, up, down projections |
| Learning rate / schedule | 2e-4 / cosine |
| Epochs / batch size | 3 / 8 |
| Trainable parameters | 8,798,208 of 502,830,976 (1.75%) |
| Training time / final loss | 276 s / 0.141 (from 0.771 at step 25) |

## Fine-tuned results and failures
Exact match 90.00% (standard 79/80 = 98.8%, edge 2/10 = 20.0%), field accuracy 96.39%, valid JSON 100%.

The failure cell below lists 9 mismatches. One of them (`'Deep Learning'` vs `\"Deep Learning\"`) is only a quote-style difference, not a meaning error. Three examples analysed in the report:
1. "Actually don't remind me..." -> predicted `reminder`, gold `cancel_reminder` (no cancel examples in training).
2. "Ping me in 2 hours..." -> predicted `tomorrow`/`14:00`, gold `today`/`unknown` (no clock, no relative or `unknown` times in training).
3. "...message Priya about notes..." -> task cut to `message Priya` (longer unseen task phrase).

**Limitations:** test data comes from the same template family as training, labels such as EOD = 17:00 are my own choices, and the edge set has only 10 examples, so 20% is a rough figure.

## Memory estimate for fine-tuning a 7B model (7e9 parameters)
Assumptions: mixed precision with Adam, short sequences, gradient checkpointing, activations about 2-4 GB.
- **(a) Full fine-tuning:** fp16 weights 2 B + fp16 grads 2 B + fp32 master weights 4 B + Adam moments 8 B = 16 B/param -> 7e9 x 16 = **112 GB** + activations.
- **(b) LoRA:** frozen fp16 weights 7e9 x 2 = 14 GB; adapters ~1% of params = 70e6 x 16 B ~ 1.1 GB; activations 2-4 GB -> **about 17-19 GB**.
- **(c) QLoRA:** 4-bit weights 7e9 x 0.5 = 3.5 GB + ~0.5 GB quantisation constants + adapters ~1.1 GB + activations 2-4 GB -> **about 7-9 GB**.